<a href="https://colab.research.google.com/github/Soljafree60/git-work/blob/main/08_PINN_FDM_Maturity_Sensitivity_CPUT_Harvard_SUBMISSION_READY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Time-to-Maturity Sensitivity Experiment
## Analytical Black–Scholes vs Crank–Nicolson FDM vs Improved PINN

This notebook varies only
$$
T\in\{0.5,1.0,2.0\}
$$
while keeping
$$
S_0=100,\;K=100,\;r=0.05,\;\sigma=0.20,\;S_{\max}=200
$$
fixed. For each maturity, a fresh PINN is trained for 10,000 epochs using the improved learning-rate schedule and best-model checkpointing. FDM and PINN are then evaluated on the same 401-point asset-price grid.


## 1. Imports and reproducibility


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import time
from scipy.stats import norm
from scipy.linalg import solve_banded

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.keras.backend.set_floatx("float32")
print("TensorFlow version:", tf.__version__)


## 2. Parameters and maturity scenarios


In [ ]:
S0=100.0
K=100.0
r=0.05
sigma=0.20
S_max=200.0
T_values=[0.5,1.0,2.0]
FDM_N=100
FDM_M=100
N_f=10_000
N_b=1_000
N_T=1_000
EPOCHS=10_000
lambda_f=lambda_B=lambda_T=1.0
LR_1=1e-3
LR_2=5e-4
LR_3=1e-4
print("Maturity values:",T_values)


## 3. Analytical Black–Scholes solution


In [ ]:
def black_scholes_call(S,K,r,sigma,T,t=0.0):
    scalar_input=np.isscalar(S)
    S=np.asarray(S,dtype=float)
    tau=T-t
    if tau<=0:
        result=np.maximum(S-K,0.0)
        return float(result) if scalar_input else result
    result=np.zeros_like(S,dtype=float)
    pos=S>0
    Sp=S[pos]
    d1=(np.log(Sp/K)+(r+0.5*sigma**2)*tau)/(sigma*np.sqrt(tau))
    d2=d1-sigma*np.sqrt(tau)
    result[pos]=Sp*norm.cdf(d1)-K*np.exp(-r*tau)*norm.cdf(d2)
    return float(result) if scalar_input else result


### Why I used the analytical Black–Scholes function

The analytical European call solution is used as the benchmark because the Black–Scholes model has a closed-form solution under the assumptions used in this study (Black & Scholes, 1973; Hull, 2018). This allowed the numerical FDM and PINN solutions to be evaluated against a known reference rather than against one another.

The implementation handles $S=0$ and maturity separately to avoid undefined expressions such as $\log(0)$ or division by zero. These are numerical safeguards; they do not alter the Black–Scholes model.


## 4. Crank–Nicolson solver


In [ ]:
def crank_nicolson_call(S_max,K,r,sigma,T,M,N):
    dt=T/M
    S=np.linspace(0.0,S_max,N+1)
    V=np.maximum(S-K,0.0)
    i=np.arange(1,N)
    a=0.25*dt*(sigma**2*i**2-r*i)
    b=-0.5*dt*(sigma**2*i**2+r)
    c=0.25*dt*(sigma**2*i**2+r*i)
    A=np.zeros((3,N-1))
    A[0,1:]=-c[:-1]
    A[1,:]=1.0-b
    A[2,:-1]=-a[1:]
    for n in range(M):
        tau_new=(n+1)*dt
        rhs=a*V[:-2]+(1.0+b)*V[1:-1]+c*V[2:]
        V_left=0.0
        V_right=S_max-K*np.exp(-r*tau_new)
        rhs[0]+=a[0]*V_left
        rhs[-1]+=c[-1]*V_right
        V_inner=solve_banded((1,1),A,rhs)
        V[0]=V_left
        V[1:-1]=V_inner
        V[-1]=V_right
    return S,V


### Why the Crank–Nicolson method is used

Crank–Nicolson is an implicit finite-difference method obtained by averaging the spatial differential operator between two adjacent time levels. It is widely used for parabolic PDEs and option-pricing problems because it provides a strong balance between accuracy and numerical stability (Crank & Nicolson, 1947; Duffy, 2006).

The European call payoff is used as the initial condition in time-to-maturity coordinates, and the Black–Scholes boundary conditions are imposed at $S=0$ and $S=S_{\max}$.


## 5. PINN architecture and maturity-dependent scaling


In [ ]:
def scale_inputs(S,t,T_value):
    return 2.0*S/S_max-1.0, 2.0*t/T_value-1.0

def model_prediction(model,S,t,T_value):
    Ss,ts=scale_inputs(S,t,T_value)
    return model(tf.concat([Ss,ts],axis=1))

class PINN(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.h1=tf.keras.layers.Dense(64,activation="tanh",kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED+1))
        self.h2=tf.keras.layers.Dense(64,activation="tanh",kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED+2))
        self.h3=tf.keras.layers.Dense(64,activation="tanh",kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED+3))
        self.h4=tf.keras.layers.Dense(64,activation="tanh",kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED+4))
        self.out=tf.keras.layers.Dense(1,kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED+5))
    def call(self,x):
        x=self.h1(x); x=self.h2(x); x=self.h3(x); x=self.h4(x)
        return self.out(x)


### Why I scaled the inputs

The asset price and time variables have different numerical ranges. Scaling them to approximately $[-1,1]$ improves the conditioning of the neural-network optimisation and is compatible with the use of `tanh` activation functions. PINNs are sensitive to optimisation and gradient behaviour, so numerical scaling can materially improve training stability (Karniadakis et al., 2021; Wang et al., 2021).

The PDE derivatives are still taken with respect to the physical variables $S$ and $t$. TensorFlow applies the chain rule through the scaling transformation automatically.


## 6. Shared spatial points and time fractions


In [ ]:
rng=np.random.default_rng(SEED)
S_f_np=rng.uniform(0.0,S_max,size=(N_f,1)).astype(np.float32)
S_T_np=rng.uniform(0.0,S_max,size=(N_T,1)).astype(np.float32)
t_f_fraction_np=rng.uniform(0.0,1.0,size=(N_f,1)).astype(np.float32)
t_b_fraction_np=rng.uniform(0.0,1.0,size=(N_b,1)).astype(np.float32)
S_f=tf.constant(S_f_np)
S_T=tf.constant(S_T_np)
S_left=tf.zeros((N_b,1),dtype=tf.float32)
S_right=tf.ones((N_b,1),dtype=tf.float32)*S_max
print("Interior points:",S_f.shape)


### Why I generated collocation, boundary and terminal points

PINNs do not require a conventional labelled training dataset for the interior of the domain. Instead, collocation points are sampled in the $(S,t)$ domain and the governing PDE is enforced through its residual (Raissi et al., 2019).

Separate boundary and terminal samples are required because the Black–Scholes PDE alone does not uniquely determine the European call solution. The boundary conditions and maturity payoff supply the additional constraints needed to identify the correct solution.


## 7. Common evaluation grid and metrics


In [ ]:
S_common=np.linspace(0.0,S_max,401)
S_common_tf=tf.constant(S_common.reshape(-1,1),dtype=tf.float32)

def calculate_metrics(prediction,exact):
    error=prediction-exact
    mse=np.mean(error**2)
    rmse=np.sqrt(mse)
    l2=np.linalg.norm(error,ord=2)
    max_abs=np.max(np.abs(error))
    return error,mse,rmse,l2,max_abs


### Why several error metrics are used

No single error measure completely describes a numerical solution. MSE and RMSE summarise average discrepancy across the evaluation grid, the $L_2$ norm measures the total magnitude of the error vector, and maximum absolute error identifies the worst local deviation.

Using several metrics prevents the comparison from depending only on the price at $S=100$. The values produced are the author's own computations.


## 8. Train one PINN for a specified maturity


In [ ]:
def train_pinn_for_T(T_value):
    t_f=tf.constant(t_f_fraction_np*T_value,dtype=tf.float32)
    t_b=tf.constant(t_b_fraction_np*T_value,dtype=tf.float32)
    t_T=tf.ones((N_T,1),dtype=tf.float32)*T_value

    V_left=tf.zeros((N_b,1),dtype=tf.float32)
    V_right=S_max-K*tf.exp(-r*(T_value-t_b))
    V_T=tf.maximum(S_T-K,0.0)
    U_left=V_left/K; U_right=V_right/K; U_T=V_T/K

    model=PINN()
    _=model_prediction(model,tf.constant([[S0]],dtype=tf.float32),tf.constant([[0.5*T_value]],dtype=tf.float32),T_value)

    def pde_residual(S,t):
        with tf.GradientTape(persistent=True) as tape2:
            tape2.watch(S); tape2.watch(t)
            with tf.GradientTape(persistent=True) as tape1:
                tape1.watch(S); tape1.watch(t)
                u=model_prediction(model,S,t,T_value)
            u_S=tape1.gradient(u,S); u_t=tape1.gradient(u,t)
        u_SS=tape2.gradient(u_S,S)
        del tape1,tape2
        return u_t+0.5*sigma**2*S**2*u_SS+r*S*u_S-r*u

    def total_loss():
        res=pde_residual(S_f,t_f)
        lpde=tf.reduce_mean(tf.square(res))
        ul=model_prediction(model,S_left,t_b,T_value)
        ur=model_prediction(model,S_right,t_b,T_value)
        lb=tf.reduce_mean(tf.square(tf.concat([ul,ur],0)-tf.concat([U_left,U_right],0)))
        ut=model_prediction(model,S_T,t_T,T_value)
        lt=tf.reduce_mean(tf.square(ut-U_T))
        return lambda_f*lpde+lambda_B*lb+lambda_T*lt,lpde,lb,lt

    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_1)
    @tf.function(reduce_retracing=True)
    def train_step():
        with tf.GradientTape() as tape:
            total,lpde,lb,lt=total_loss()
        grads=tape.gradient(total,model.trainable_variables)
        optimizer.apply_gradients([(g,v) for g,v in zip(grads,model.trainable_variables) if g is not None])
        return total,lpde,lb,lt

    history=[]; best_loss=np.inf; best_epoch=0; best_weights=None
    start=time.perf_counter()
    for epoch in range(1,EPOCHS+1):
        if epoch==3001: optimizer.learning_rate.assign(LR_2)
        if epoch==7001: optimizer.learning_rate.assign(LR_3)
        total,lpde,lb,lt=train_step()
        val=float(total.numpy()); history.append(val)
        if val<best_loss:
            best_loss=val; best_epoch=epoch; best_weights=model.get_weights()
        if epoch==1 or epoch%1000==0:
            print(f"T={T_value:.1f} | Epoch {epoch:5d} | LR={optimizer.learning_rate.numpy():.1e} | Total={total.numpy():.6e} | PDE={lpde.numpy():.6e} | B={lb.numpy():.6e} | Tloss={lt.numpy():.6e}")
    training_time=time.perf_counter()-start
    model.set_weights(best_weights)
    return {'model':model,'best_epoch':best_epoch,'best_loss':best_loss,'training_time':training_time,'loss_history':history}


### Why I used automatic differentiation for the PDE residual

The PINN solution is constrained by the Black–Scholes PDE. TensorFlow automatic differentiation is therefore used to obtain

$$
u_t,\qquad u_S,\qquad u_{SS}.
$$

These derivatives are substituted directly into the differential equation to form the residual. Minimising the residual at collocation points is the central physics-informed mechanism that allows the network to learn the PDE solution without labelled interior solution values (Raissi et al., 2019; Karniadakis et al., 2021).

Nested gradient tapes are required because $u_{SS}$ is a second derivative.


## 9. Run all maturity scenarios


In [ ]:
maturity_results=[]
training_histories={}

for T_value in T_values:
    print("\n"+"="*70)
    print(f"STARTING MATURITY EXPERIMENT: T = {T_value:.1f}")
    print("="*70)

    V_exact=black_scholes_call(S_common,K,r,sigma,T_value,t=0.0)
    exact_price=black_scholes_call(S0,K,r,sigma,T_value,t=0.0)

    fdm_times=[]
    for _ in range(20):
        st=time.perf_counter()
        S_fdm,V_fdm=crank_nicolson_call(S_max,K,r,sigma,T_value,FDM_M,FDM_N)
        fdm_times.append(time.perf_counter()-st)
    fdm_runtime=np.median(fdm_times)
    V_fdm_common=np.interp(S_common,S_fdm,V_fdm)
    fdm_price=np.interp(S0,S_common,V_fdm_common)
    _,fdm_mse,fdm_rmse,fdm_l2,fdm_max=calculate_metrics(V_fdm_common,V_exact)
    fdm_abs=abs(fdm_price-exact_price)
    fdm_rel=fdm_abs/exact_price*100.0

    pout=train_pinn_for_T(T_value)
    model=pout['model']; training_histories[T_value]=pout['loss_history']
    t0=tf.zeros((len(S_common),1),dtype=tf.float32)
    _=model_prediction(model,S_common_tf,t0,T_value).numpy()
    inf=[]
    for _ in range(100):
        st=time.perf_counter(); _=(K*model_prediction(model,S_common_tf,t0,T_value)).numpy(); inf.append(time.perf_counter()-st)
    pinn_runtime=np.median(inf)
    V_pinn=(K*model_prediction(model,S_common_tf,t0,T_value)).numpy().reshape(-1)
    pinn_price=np.interp(S0,S_common,V_pinn)
    _,pinn_mse,pinn_rmse,pinn_l2,pinn_max=calculate_metrics(V_pinn,V_exact)
    pinn_abs=abs(pinn_price-exact_price)
    pinn_rel=pinn_abs/exact_price*100.0

    maturity_results.append({'Maturity T':T_value,'Method':'Crank-Nicolson FDM','Analytical Price':exact_price,'Price at S=100':fdm_price,'Absolute Price Error':fdm_abs,'Relative Price Error (%)':fdm_rel,'MSE':fdm_mse,'RMSE':fdm_rmse,'L2 Error':fdm_l2,'Maximum Abs Error':fdm_max,'Solve / Inference Runtime (s)':fdm_runtime,'Training Runtime (s)':np.nan,'Best Epoch':np.nan,'Best Total Loss':np.nan})
    maturity_results.append({'Maturity T':T_value,'Method':'Improved PINN','Analytical Price':exact_price,'Price at S=100':pinn_price,'Absolute Price Error':pinn_abs,'Relative Price Error (%)':pinn_rel,'MSE':pinn_mse,'RMSE':pinn_rmse,'L2 Error':pinn_l2,'Maximum Abs Error':pinn_max,'Solve / Inference Runtime (s)':pinn_runtime,'Training Runtime (s)':pout['training_time'],'Best Epoch':pout['best_epoch'],'Best Total Loss':pout['best_loss']})
    print(f"Completed T = {T_value:.1f}")

maturity_df=pd.DataFrame(maturity_results)
maturity_df


### Why I stored the results in a DataFrame

A pandas DataFrame is used to keep prices, errors, runtimes and model settings in a structured table. This makes it easier to compare methods consistently, export results, and reproduce the tables used in Chapter 4 (McKinney, 2010).



## 10. Plots


In [ ]:
fdm_rows=maturity_df[maturity_df['Method']=='Crank-Nicolson FDM'].sort_values('Maturity T')
pinn_rows=maturity_df[maturity_df['Method']=='Improved PINN'].sort_values('Maturity T')

plt.figure(figsize=(10,6))
plt.plot(fdm_rows['Maturity T'],fdm_rows['Analytical Price'],marker='o',label='Analytical Black-Scholes')
plt.plot(fdm_rows['Maturity T'],fdm_rows['Price at S=100'],marker='o',linestyle='--',label='Crank-Nicolson FDM')
plt.plot(pinn_rows['Maturity T'],pinn_rows['Price at S=100'],marker='o',linestyle=':',label='Improved PINN')
plt.xlabel('Time to Maturity (T)'); plt.ylabel('Call Option Price at S=100'); plt.title('Effect of Time to Maturity on European Call Price'); plt.legend(); plt.grid(True); plt.show()

plt.figure(figsize=(10,6))
plt.plot(fdm_rows['Maturity T'],fdm_rows['Absolute Price Error'],marker='o',label='FDM')
plt.plot(pinn_rows['Maturity T'],pinn_rows['Absolute Price Error'],marker='o',label='PINN')
plt.xlabel('Time to Maturity (T)'); plt.ylabel('Absolute Price Error'); plt.title('Absolute Pricing Error vs Time to Maturity'); plt.legend(); plt.grid(True); plt.show()

plt.figure(figsize=(10,6))
plt.plot(fdm_rows['Maturity T'],fdm_rows['MSE'],marker='o',label='FDM')
plt.plot(pinn_rows['Maturity T'],pinn_rows['MSE'],marker='o',label='PINN')
plt.yscale('log'); plt.xlabel('Time to Maturity (T)'); plt.ylabel('MSE (log scale)'); plt.title('MSE Sensitivity to Time to Maturity'); plt.legend(); plt.grid(True); plt.show()

plt.figure(figsize=(10,6))
plt.plot(pinn_rows['Maturity T'],pinn_rows['Training Runtime (s)'],marker='o')
plt.xlabel('Time to Maturity (T)'); plt.ylabel('PINN Training Runtime (seconds)'); plt.title('PINN Training Runtime vs Time to Maturity'); plt.grid(True); plt.show()

plt.figure(figsize=(10,6))
for Tv in T_values: plt.semilogy(training_histories[Tv],label=f'T = {Tv:.1f}')
plt.axvline(3000,linestyle='--'); plt.axvline(7000,linestyle='--')
plt.xlabel('Epoch'); plt.ylabel('Total PINN Loss'); plt.title('PINN Convergence Under Different Times to Maturity'); plt.legend(); plt.grid(True); plt.show()


### Why I used a logarithmic scale for the loss graph

PINN losses often change by several orders of magnitude during training. A logarithmic vertical axis makes both the early large losses and the later small losses visible on the same figure. The graph is used to assess convergence and identify instability or spikes in the optimisation process. Matplotlib is used for the visualisation (Hunter, 2007).


## 11. Save results


In [ ]:
maturity_df.to_csv('maturity_sensitivity_results.csv',index=False)
print('Saved: maturity_sensitivity_results.csv')


## 12. Outputs used from this experiment

The main outputs I kept from this maturity experiment were the full `maturity_df` table, the price-versus-maturity graph, the absolute-error graph, the MSE graph, the PINN training-runtime graph and the PINN convergence graph.

I also kept the printed training results for

$$
T = 0.5,\qquad T = 1.0,\qquad T = 2.0.
$$



## Results obtained from the completed time-to-maturity sensitivity study

|   Maturity T | Method             |   Analytical Price |   Price at S=100 |   Absolute Price Error |   Relative Price Error (%) |         MSE |       RMSE |   L2 Error |   Solve / Inference Runtime (s) |   Training Runtime (s) |
|-------------:|:-------------------|-------------------:|-----------------:|-----------------------:|---------------------------:|------------:|-----------:|-----------:|--------------------------------:|-----------------------:|
|          0.5 | Crank-Nicolson FDM |            6.88873 |          6.87465 |             0.0140756  |                  0.204328  | 7.12048e-06 | 0.00266842 |  0.0534351 |                      0.00478808 |                 nan    |
|          0.5 | Improved PINN      |            6.88873 |          6.84746 |             0.0412645  |                  0.599015  | 0.0085135   | 0.0922686  |  1.84768   |                      0.00926273 |                1898.76 |
|          1   | Crank-Nicolson FDM |           10.4506  |         10.4407  |             0.00987333 |                  0.0944763 | 5.83264e-06 | 0.00241509 |  0.0483621 |                      0.00441134 |                 nan    |
|          1   | Improved PINN      |           10.4506  |         10.5025  |             0.0518979  |                  0.496603  | 0.00622687  | 0.0789105  |  1.58018   |                      0.00944092 |                1890.04 |
|          2   | Crank-Nicolson FDM |           16.1268  |         16.1199  |             0.00688527 |                  0.0426946 | 4.32253e-05 | 0.0065746  |  0.131656  |                      0.00452617 |                 nan    |
|          2   | Improved PINN      |           16.1268  |         15.898   |             0.228775   |                  1.4186    | 0.031479    | 0.177423   |  3.5529    |                      0.00891299 |                1875.11 |

The call price increased over the tested maturity values under the non-dividend-paying Black–Scholes assumptions (Black & Scholes, 1973; Hull, 2018). The FDM remained more accurate across all three maturities. The PINN error increased most strongly at $T=2$.

Because $M=100$ FDM time steps and the same number of PINN collocation points were used for every maturity, this experiment is interpreted as a robustness test under a fixed computational budget. As $T$ increases, the physical FDM time step $T/M$ becomes larger and the PINN collocation points span a longer time domain (Duffy, 2006; Raissi et al., 2019).




## References — CPUT Harvard style

Black, F. & Scholes, M. 1973. The pricing of options and corporate liabilities. *Journal of Political Economy*, 81(3):637-654. DOI: 10.1086/260062.

Crank, J. & Nicolson, P. 1947. A practical method for numerical evaluation of solutions of partial differential equations of the heat-conduction type. *Proceedings of the Cambridge Philosophical Society*, 43(1):50-67. DOI: 10.1017/S0305004100023197.

Duffy, D.J. 2006. *Finite difference methods in financial engineering: A partial differential equation approach*. Chichester: John Wiley & Sons. DOI: 10.1002/9781118673447.

Harris, C.R., Millman, K.J., van der Walt, S.J., Gommers, R., Virtanen, P., Cournapeau, D., Wieser, E., Taylor, J., Berg, S., Smith, N.J., Kern, R., Picus, M., Hoyer, S., van Kerkwijk, M.H., Brett, M., Haldane, A., del Río, J.F., Wiebe, M., Peterson, P., Gérard-Marchant, P., Sheppard, K., Reddy, T., Weckesser, W., Abbasi, H., Gohlke, C. & Oliphant, T.E. 2020. Array programming with NumPy. *Nature*, 585:357-362. DOI: 10.1038/s41586-020-2649-2.

Hull, J.C. 2018. *Options, futures, and other derivatives*. 10th ed. Harlow: Pearson.

Hunter, J.D. 2007. Matplotlib: A 2D graphics environment. *Computing in Science & Engineering*, 9(3):90-95. DOI: 10.1109/MCSE.2007.55.

Karniadakis, G.E., Kevrekidis, I.G., Lu, L., Perdikaris, P., Wang, S. & Yang, L. 2021. Physics-informed machine learning. *Nature Reviews Physics*, 3(6):422-440. DOI: 10.1038/s42254-021-00314-5.

McKinney, W. 2010. Data structures for statistical computing in Python. In: van der Walt, S. & Millman, J. eds. *Proceedings of the 9th Python in Science Conference*. Austin, TX: SciPy, 56-61. DOI: 10.25080/Majora-92bf1922-00a.

Raissi, M., Perdikaris, P. & Karniadakis, G.E. 2019. Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations. *Journal of Computational Physics*, 378:686-707. DOI: 10.1016/j.jcp.2018.10.045.

Virtanen, P., Gommers, R., Oliphant, T.E., Haberland, M., Reddy, T., Cournapeau, D., Burovski, E., Peterson, P., Weckesser, W., Bright, J., van der Walt, S.J., Brett, M., Wilson, J., Millman, K.J., Mayorov, N., Nelson, A.R.J., Jones, E., Kern, R., Larson, E., Carey, C.J., Polat, İ., Feng, Y., Moore, E.W., VanderPlas, J., Laxalde, D., Perktold, J., Cimrman, R., Henriksen, I., Quintero, E.A., Harris, C.R., Archibald, A.M., Ribeiro, A.H., Pedregosa, F., van Mulbregt, P. & SciPy 1.0 Contributors. 2020. SciPy 1.0: Fundamental algorithms for scientific computing in Python. *Nature Methods*, 17:261-272. DOI: 10.1038/s41592-019-0686-2.

Wang, S., Teng, Y. & Perdikaris, P. 2021. Understanding and mitigating gradient flow pathologies in physics-informed neural networks. *SIAM Journal on Scientific Computing*, 43(5):A3055-A3081. DOI: 10.1137/20M1318043.
